### Preprocessing
Preprocessing Plan: 
* Load in the data
* Remove duplicate rows 
* Handle empty and very short reviews 
* Join Product Metadata
* Clean Review Text
* Select Relevant Columns
* Save the Processed Dataset

In [1]:
import pandas as pd

beauty_reviews = pd.read_pickle("beauty_reviews.pkl")
beauty_meta = pd.read_pickle("beauty_meta.pkl")

In [2]:
df = beauty_reviews.copy()

# Make a copy in order to not alter the original, unprocessed notebook
print("Reviews:", df.shape)
print("Metadata:", beauty_meta.shape)

df.head()

Reviews: (701528, 11)
Metadata: (112590, 14)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_length
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True,300
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True,235
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True,25
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True,14
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True,7


### Identify Duplicate Rows

In [3]:
duplicate_cols = ["user_id", "parent_asin", "timestamp", "rating", "text"]

print("Before:", len(df))
df = df.drop_duplicates(subset=duplicate_cols).copy()

print("After:", len(df))
print("Removed:", len(beauty_reviews) - len(df))

Before: 701528
After: 694252
Removed: 7276


### Handle Short Reviews
This project focuses on identifying customer complaints and characteristics, so the data should primarily consist of informative reviews. 

In [5]:
# Investigate review length
df["review_length"] = df["text"].str.len()

print(df["review_length"].describe())

count    694252.000000
mean        173.102657
std         247.168573
min           0.000000
25%          44.000000
50%         102.000000
75%         209.000000
max       14989.000000
Name: review_length, dtype: float64


In [11]:
# Random sample of 30 short reviews (reviews less than 30 characters)
short_reviews = df[df["review_length"] < 30]
short_reviews[["text", "rating"]].sample(n=min(20, len(short_reviews)),random_state=42)

,text,rating
665843,"worth the price , good deal",4
89185,Loved it!!,5
448981,Gorgeous!,5
575232,Good,3
547909,Met expectations,5
200828,High quality low price,5
247122,Good toothbrush thank you,5
419164,Love it!,5
204502,No sensitive skin,1
631564,great price.,5


These reviews look informative despite their short length, so for now they will be kept. 

## Join Reviews with Product Metadata

In [12]:
beauty_meta.columns.tolist()

['main_category',
 'title',
 'average_rating',
 'rating_number',
 'features',
 'description',
 'price',
 'images',
 'videos',
 'store',
 'categories',
 'details',
 'parent_asin',
 'bought_together']

In [14]:
product_info = beauty_meta[["parent_asin", "title", "store", "categories"]].drop_duplicates(subset="parent_asin")

df = df.merge(product_info, on="parent_asin", how="left")

print("Reviews after merge:", len(df))
df.columns.tolist()
# df[["text", "rating", "title", "store", "categories"]].head()

Reviews after merge: 694252


['rating',
 'title_x',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'review_length',
 'title_y',
 'store_x',
 'categories_x',
 'title',
 'store_y',
 'categories_y']

In [16]:
# Fix Column Names after Merging
df = df.rename(columns={"title_x": "review_title", "title_y": "product_title", "store_x": "store", "categories_x": "categories"})

df[["text", "rating", "review_title", "product_title", "store", "categories"]].head()

,text,rating,review_title,product_title,store,categories
0,This spray is really nice. It smells really go...,5,Such a lovely scent but not overpowering.,Herbivore - Natural Sea Mist Texturizing Salt ...,HERBIVORE,[]
1,"This product does what I need it to do, I just...",4,Works great but smells a little weird.,All Natural Vegan Dry Shampoo Powder - Eco Fri...,Two Goats Apothecary,[]
2,"Smells good, feels great!",5,Yes!,New Road Beauty - Creamsicle - Variety 3 Pack ...,New Road Beauty,[]
3,Felt synthetic,1,Synthetic feeling,muaowig Ombre Body Wave Bundles 1B Grey Human ...,muaowig,[]
4,Love it,5,A+,Yinhua Electric Nail Drill Kit Portable Profes...,Yinhua,[]


In [17]:
print("Total reviews:", len(df))
print("Missing product titles:", df["product_title"].isna().sum())
print("Missing stores:", df["store"].isna().sum())

Total reviews: 694252
Missing product titles: 0
Missing stores: 49380


In [20]:
df["clean_text"] = (
    df["text"]
    # Replace repeated white spaces with a singular space
    .str.replace(r"\s+", " ", regex=True)
    # Removes spaces at the beginning and end of the text
    .str.strip()
)

# Check for empty reviews
print("Empty reviews:", (df["clean_text"] == "").sum())

df[["text", "clean_text"]].sample(5, random_state=42)

Empty reviews: 705


,text,clean_text
362775,All shimmer/sparkles... I’ve used Essie all my...,All shimmer/sparkles... I’ve used Essie all my...
653008,"I reseived it, used it. Now I'm healing burns ...","I reseived it, used it. Now I'm healing burns ..."
110194,"This face oil is perfect for my skin type,it i...","This face oil is perfect for my skin type,it i..."
176993,Very cute! Love it!,Very cute! Love it!
65532,I get my nails did every 2-4 weeks. The variou...,I get my nails did every 2-4 weeks. The variou...


There are only 705 empty rows which is a very small portion of our dataset (0.1%), so they will be removed.

In [21]:
df = df[df["clean_text"] != ""].copy()

### Check the distribution of Sentiment Labels
Initially, star ratings will be used to assess the sentiment of reviews:
* 1-2 Stars: Negative Sentiment
* 3 Stars: Neutral Sentiment
* 4-5 Stars: Positive Sentiment

In [23]:
# Create Labels for Sentiment
df["sentiment"] = df["rating"].map({1: "Negative", 2: "Negative", 3: "Neutral", 4: "Positive", 5: "Positive"})

# Check the distributions of Sentiment Labels
print(df["sentiment"].value_counts())

sentiment
Positive    494518
Negative    143333
Neutral      55696
Name: count, dtype: int64


In [24]:
# Check percentages of Sentiment Labels
print(df["sentiment"].value_counts(normalize=True) * 100)

sentiment
Positive    71.302738
Negative    20.666660
Neutral      8.030602
Name: proportion, dtype: float64


In [25]:
df.columns

Index(['rating', 'review_title', 'text', 'images', 'asin', 'parent_asin',
       'user_id', 'timestamp', 'helpful_vote', 'verified_purchase',
       'review_length', 'product_title', 'store', 'categories', 'title',
       'store_y', 'categories_y', 'clean_text', 'sentiment'],
      dtype='str')

In [26]:
# Keep only columns we need; get rid of images, verified_purchase, timestamp, helpful_vote, store_y, categories_y
columns = ["user_id", "parent_asin", "product_title", "store", "categories", "rating", "sentiment", "text", "clean_text", "review_length"]

clean_df = df[columns].copy()

### Key Findings
* 7,276 Duplicate Records were removed, leaving 694,252 reviews
* 705 of the reviews were empty reviews; those were removed
* After inspection, short reviews (reviews with length shorter than 30 characters) were kept in the data, as they still contained insightful information
* Reviews were merged with meta data information, though 49,380 reviews are missing the store attribute 
* Review texts were cleaned by removing repeated whitespace sequences and line breaks
* Reviews were assigned a sentiment label based off of the star rating:
    * 71% had positive sentiment
    * 8% had neutral sentiment
    * 23% had negative sentiment

In [27]:
# Save and Export the Cleaned Dataset
clean_df.to_pickle("beauty_clean.pkl")

print("Saved:", clean_df.shape)

Saved: (693547, 10)
